# 🏭 Smart Factory — Pipeline Analisi Incidenti v3
**Università La Sapienza — Assignment 1**

Analisi audio multi-fonte con Gemini 2.5 Flash → Ontologia → Excel

---
## ⚙️ Modalità disponibili

| Modalità | Descrizione | Output Excel |
|---|---|---|
| **MODALITÀ A** — Singolo audio | Ogni audio viene analizzato separatamente da Gemini → 1 tabella per audio | `report_singolo_NOMEAUDIO.xlsx` per ciascun file |
| **MODALITÀ B** — Fusione | Fonde i risultati della Modalità A → 1 tabella unificata, **senza nuove chiamate API** | `report_multiaudio.xlsx` |
| **MODALITÀ C** — Analisi Qualità | Ogni audio viene analizzato N volte → tabelle multi-estrazione + foglio metriche qualità | `report_qualita_NOMEAUDIO.xlsx` per ciascun file |

> **Ordine di esecuzione consigliato:**
> - Modalità A → Cella 12, poi Cella 13
> - Modalità B → Cella 14 (richiede Modalità A)
> - Modalità C → Cella 15 (configurazione N), poi Cella 16 (trascrizione, opzionale), poi Cella 17 (analisi), poi Cella 18 (salvataggio)


In [1]:
# =====================================================================
# CELLA 1: INSTALLAZIONE — esegui UNA SOLA VOLTA, poi riavvia il kernel
# =====================================================================
%pip install google-generativeai pandas openpyxl scikit-learn scipy


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# =====================================================================
# CELLA 2: IMPORT
# =====================================================================
import os
import re
import time
import json
import math
import itertools
import pandas as pd
import google.generativeai as genai

# formattazione visiva di excel
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter
from IPython.display import display, Markdown

# metriche qualità
from sklearn.metrics import cohen_kappa_score
from collections import Counter

print('✅ Import completati correttamente.')


C:\Users\flavc\AppData\Local\Temp\ipykernel_23456\4088727717.py:11: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


✅ Import completati correttamente.


In [3]:
# =====================================================================
# CELLA 3: CONFIGURAZIONE
# SICUREZZA: imposta la chiave come variabile d'ambiente:
#   Windows:   set GEMINI_API_KEY=la_tua_chiave
#   Mac/Linux: export GEMINI_API_KEY=la_tua_chiave
# In alternativa scrivi la chiave qui solo per test locali.
# =====================================================================

GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]
genai.configure(api_key=GEMINI_API_KEY)

# ---------------------------------------------------------------
# 🎯 LISTA FILE AUDIO — aggiungi o rimuovi percorsi liberamente!
#    Puoi usare percorsi assoluti (r"C:\...") o relativi ("audio.mp3")
#    Formati supportati: .mp3, .wav, .m4a, .ogg, .flac
# ---------------------------------------------------------------
FILE_AUDIO = [
    "Elena RSPP.mp3",
    "Marco Capolinea.mp3",
    "Martina Operatrice.mp3",
    # Aggiungi altri file qui sotto se necessario:
    # r"percorso\audio4.mp3",
    # r"percorso\audio5.wav",
]

# Cartella di output (verrà creata se non esiste)
OUTPUT_DIR = "output_report"
os.makedirs(OUTPUT_DIR, exist_ok=True)

#il ciclo stampa per ogni file uno stato di verifica se esiste o meno, così da avere un riscontro immediato prima di procedere con l'elaborazione
print(f'✅ Configurazione completata. {len(FILE_AUDIO)} file audio configurati.')
for i, f in enumerate(FILE_AUDIO, 1):
    esiste = "✅" if os.path.exists(f) else "❌ NON TROVATO"
    print(f"   Audio {i}: {os.path.basename(f)} — {esiste}")

✅ Configurazione completata. 3 file audio configurati.
   Audio 1: Elena RSPP.mp3 — ✅
   Audio 2: Marco Capolinea.mp3 — ✅
   Audio 3: Martina Operatrice.mp3 — ✅


In [4]:
# =====================================================================
# CELLA 4: PROMPT DI SISTEMA
# Due prompt distinti per le due modalità.
# =====================================================================

# --- STRUTTURA JSON CONDIVISA (ontologia) ---
_SCHEMA_JSON = """
{
  "Evento": {
    "id_evento": "INC-AAAA-NNN",
    "Data_Ora": "GG/MM/AAAA HH:MM",
    "Tipo_Evento": "Incidente | Near Miss | Guasto Tecnico | Manutenzione",
    "Gravita_Stimata": "Bassa | Media | Alta | Critica | Fatale",
    "Prima_dell_evento": "Descrizione stato macchina e lavoratore",
    "Trigger_Evento_Principale": "Causa scatenante immediata",
    "Dinamica_Sintesi": "Riassunto neutro e oggettivo",
  },
  "Luoghi_e_Mezzi": {
    "Reparto": "Nome univoco del reparto (es. Reparto Stampa, Linea 3). Scrivi N/D se non menzionato.",
    "Area_Coinvolta": "Zona specifica all'interno del reparto (es. Zona pressa, Uscita nastro). Scrivi N/D se non menzionata.",
    "Macchinario_Coinvolto": "Nome tecnico del macchinario (es. Pressa idraulica P200, Tornio CNC). Scrivi N/D se non menzionato.",
    "Codice_Macchinario": "Codice o targa del macchinario se citato, altrimenti N/D.",
    "Condizioni_Ambientali": "Condizioni fisiche dell'ambiente al momento dell'evento (es. pavimento bagnato, scarsa illuminazione). Scrivi N/D se non menzionate."
  },
  "Analisi_Cause": {
    "Causa_Tecnica": "",
    "Fattore_Umano": "",
    "Fattore_Ambientale": "",
    "Causa_Radice": "Root cause"
  },
  "Conseguenze": {
    "Conseguenza_Fisica_Immediata": "",
    "Parte_Corpo_Lesa": "",
    "Giorni_Inabilita_Stimati": "",
    "Conseguenza_Materiale_Aziendale": "",
    "Fermo_Produzione_Ore": ""
  },
  "Azioni_Correttive": {
    "Procedura_Corretta_Violata": "",
    "DPI_Mancanti_o_Inadeguati": "",
    "Azione_Correttiva_Immediata": "",
    "Azione_Correttiva_da_Implementare": "",
    "Responsabile_Azione": "",
    "Scadenza_Azione": ""
  },
  "Persone_Coinvolte": [
    {
      "Nome": "",
      "Ruolo_Aziendale": "",
      "Coinvolgimento": "Vittima | Testimone | RSPP | Supervisore",
      "Danno_Riportato": "",
      "Affidabilita_Testimonianza": "Alta | Media | Bassa",
      "Note_Testimonianza": ""
    }
  ]
}
"""

# ---- PROMPT MODALITÀ A: singolo audio ----
def build_prompt_singolo(nome_file, indice_totale):
    """Genera il prompt per l'analisi di un singolo audio."""
    return f"""
Ruolo: Sei un esperto HSE (Health, Safety & Environment) e Data Extraction.
Ricevi UNA registrazione audio (file {indice_totale}: '{nome_file}') relativa a un incidente industriale.

Istruzioni:
1. Estrai tutte le informazioni contenute in questo singolo audio.
2. Per i campi non menzionati nel racconto, scrivi esattamente "N/D".
3. Non inventare o inferire informazioni non presenti nell'audio. #riduce le allucinazioni
4. Usa terminologia tecnica HSE corretta; se il parlante usa gergo, traduci.

Istruzioni specifiche per Luoghi_e_Mezzi:
- Reparto: estrai il nome esatto del reparto citato dal parlante. Se usa un soprannome o abbreviazione, riportala così com'è senza modifiche.
- Area_Coinvolta: estrai la zona precisa all'interno del reparto (es. vicino alla pressa, zona scarico). Se non citata esplicitamente, scrivi N/D.
- Macchinario_Coinvolto: estrai il nome tecnico o il nome comune usato dal parlante per il macchinario. Se usa un soprannome (es. 'la bestia', 'il tornio vecchio'), riportalo e affianca il termine tecnico se deducibile dal contesto.
- Codice_Macchinario: solo se citato esplicitamente (targa, codice alfanumerico). Altrimenti N/D.
Macchinario_Coinvolto e Area_Coinvolta non possono essere uguali

Formato output:
Restituisci ESCLUSIVAMENTE un oggetto JSON valido (nessun testo prima/dopo, nessun backtick).
Schema:
{_SCHEMA_JSON}
"""

# ---- PROMPT MODALITÀ B: N audio insieme ----
def build_prompt_multi(risultati_singoli):
    """Genera il prompt per la fusione intelligente dei JSON estratti."""
    
    # Convertiamo i risultati della Modalità A in una stringa leggibile per Gemini
    testi_report = ""
    for i, r in enumerate(risultati_singoli):
        testi_report += f"\n--- REPORT AUDIO {i+1} ({r['nome']}) ---\n"
        testi_report += json.dumps(r["dati"], indent=2, ensure_ascii=False)

    return f"""
Ruolo: Sei un Senior HSE Specialist esperto in investigazione di incidenti.
Hai analizzato singolarmente 3 testimonianze. Ora devi redigere il RAPPORTO FINALE UNIFICATO.

ECCO I REPORT PARZIALI DA UNIRE:
{testi_report}

ISTRUZIONI RIGOROSE PER LA SINTESI:
1. RISPOSTA UNICA: Per ogni campo (es. Macchinario, Trigger), fornisci una sola descrizione sintetica. NON usare " | " o elenchi di versioni diverse.
2. PRIORITÀ TECNICA: Se le fonti usano nomi diversi, usa la terminologia tecnica del RSPP o quella normativa HSE più corretta.
3. RICONCILIAZIONE: Se una fonte è incompleta e l'altra è dettagliata, usa la versione dettagliata per arricchire il dato unico.
4. DEDUPLICAZIONE PERSONE: Crea una lista unica di 'Persone_Coinvolte'. Fondi le informazioni (ruolo, danno) in un unico oggetto per persona.
5. Se un dato è totalmente assente in tutti i report, scrivi "N/D".

Formato output:
Restituisci ESCLUSIVAMENTE un oggetto JSON valido secondo lo schema:
{_SCHEMA_JSON}
"""

print("✅ Prompt Modalità B (AI Fusion) configurato.")



✅ Prompt Modalità B (AI Fusion) configurato.


In [5]:
# =====================================================================
# CELLA 5: UTILITY GEMINI — upload, attesa, cleanup
# =====================================================================

def _upload_e_attendi(percorsi):
    """Carica una lista di file su Gemini Files API e attende che siano pronti.
    Restituisce la lista di oggetti file Gemini."""
    file_caricati = []
    for percorso in percorsi:
        if not os.path.exists(percorso):
            raise FileNotFoundError(f"❌ File non trovato: '{percorso}'")
        print(f"   ⬆️  Upload: {os.path.basename(percorso)}")
        file_caricati.append(genai.upload_file(path=percorso))

    print("   ⏳ Attendo elaborazione", end="")
    for i, f in enumerate(file_caricati):
        while f.state.name == "PROCESSING":
            print(".", end="", flush=True)
            time.sleep(2)
            file_caricati[i] = genai.get_file(f.name)
            f = file_caricati[i]
    print(" ✅")
    return file_caricati


def _chiedi_gemini(prompt, file_gemini):
    """Invia prompt + file a Gemini e restituisce il JSON parsato."""
    model = genai.GenerativeModel(
        model_name="gemini-2.5-flash",
        generation_config={"response_mime_type": "application/json"}
    )
    risposta = model.generate_content([prompt] + file_gemini)
    testo = re.sub(r"```json\s*|\s*```", "", risposta.text).strip()
    return json.loads(testo)


def _cleanup(file_gemini):
    """Elimina i file dal cloud Gemini dopo l'uso."""
    for f in file_gemini:
        try:
            genai.delete_file(f.name)
        except Exception:
            pass

print("✅ Utility Gemini pronte.")

✅ Utility Gemini pronte.


In [6]:
# =====================================================================
# CELLA 6: MODALITÀ A — analisi singola per ciascun audio
#ogni file viene processato uno alla volta in sequenza, con un report dedicato per ciascuno. Questa modalità è più lenta ma riduce le interferenze tra fonti diverse.
# =====================================================================

def analizza_singoli(lista_percorsi):
    """Analizza ogni audio INDIVIDUALMENTE.
    Ritorna una lista di dict: [{"percorso": ..., "dati": {...}}, ...]"""
    risultati = []
    n = len(lista_percorsi)
    for idx, percorso in enumerate(lista_percorsi, 1):
        nome = os.path.basename(percorso)
        print(f"\n🎙️ [{idx}/{n}] Analisi singola: '{nome}'")
        file_caricati = _upload_e_attendi([percorso])
        print(f"   🧠 Estrazione dati...")
        prompt = build_prompt_singolo(nome, idx)
        dati = _chiedi_gemini(prompt, file_caricati)
        _cleanup(file_caricati)
        dati["_meta"] = {"fonte": nome, "indice": idx, "modalita": "singolo"}
        risultati.append({"percorso": percorso, "nome": nome, "dati": dati})
        print(f"   ✅ Completato.")
    print(f"\n✅ Modalità A completata: {n} audio analizzati singolarmente.")
    return risultati

print("✅ Funzione modalità A pronta.")

✅ Funzione modalità A pronta.


In [7]:
# =====================================================================
# CELLA 7: MODALITÀ B — fusione dei dati già estratti dalla Modalità A
# Non effettua nuove chiamate a Gemini: lavora sui risultati_singoli
# già in memoria, fondendoli in un unico dict con logica Python.
# =====================================================================

def fondi_risultati_singoli(risultati_singoli):
    """Chiama Gemini per fondere i JSON ottenuti dalla Modalità A."""
    if not risultati_singoli:
        raise ValueError("Lista risultati vuota. Esegui prima la Modalità A.")

    print(f"\n🧠 Avvio fusione intelligente di {len(risultati_singoli)} fonti con Gemini...")

    # 1. Prepariamo il prompt usando la funzione della Cella 4
    prompt_fusione = build_prompt_multi(risultati_singoli)
    
    # 2. Chiamata a Gemini 1.5 Flash
    model = genai.GenerativeModel("gemini-2.5-flash")
    risposta = model.generate_content(prompt_fusione)
    
    try:
        # Pulizia backticks markdown e parsing JSON
        testo_pulito = risposta.text.strip().replace("```json", "").replace("```", "")
        return json.loads(testo_pulito)
    except Exception as e:
        print(f"❌ Errore nel parsing della fusione: {e}")
        return risultati_singoli[0]["dati"] # Fallback sul primo report in caso di errore

print("✅ Funzione fondi_risultati_singoli pronta.")

✅ Funzione fondi_risultati_singoli pronta.


In [8]:
# =====================================================================
# CELLA 8: UTILITY EXCEL — stili e funzione di formattazione
# =====================================================================

# Stili globali riutilizzabili
HEADER_FONT  = Font(bold=True, size=11, color="FFFFFF", name="Arial")
HEADER_FILL  = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
CELL_FONT    = Font(size=10, name="Arial")
ALT_FILL     = PatternFill(start_color="EBF5FB", end_color="EBF5FB", fill_type="solid")
WRAP_TOP     = Alignment(wrap_text=True, vertical="top")
CENTER       = Alignment(horizontal="center", vertical="center", wrap_text=True)
THIN         = Side(style="thin", color="BFBFBF")
BORDER       = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)


def _applica_stile(ws, larghezze=None):
    """Applica intestazioni, righe alternate, bordi e larghezze a un foglio."""
    for cell in ws[1]:
        cell.font, cell.fill, cell.alignment, cell.border = HEADER_FONT, HEADER_FILL, CENTER, BORDER
    for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):
        for cell in row:
            cell.font, cell.alignment, cell.border = CELL_FONT, WRAP_TOP, BORDER
            if row_idx % 2 == 0:
                cell.fill = ALT_FILL
    for i, col in enumerate(ws.columns):
        letter = get_column_letter(col[0].column)
        if larghezze and i < len(larghezze):
            ws.column_dimensions[letter].width = larghezze[i]
        else:
            max_len = max((len(str(c.value or "")) for c in col), default=10)
            ws.column_dimensions[letter].width = min(max(max_len * 0.85, 15), 55)
    ws.freeze_panes = "A2"
    ws.row_dimensions[1].height = 32


def _dati_a_dataframes(dati):
    """Converte il dict JSON estratto in dizionario di DataFrame (uno per sezione)."""
    ev = dati.get("Evento", {})
    lm = dati.get("Luoghi_e_Mezzi", {})
    ac = dati.get("Analisi_Cause", {})
    co = dati.get("Conseguenze", {})
    az = dati.get("Azioni_Correttive", {})
    pc = dati.get("Persone_Coinvolte", [])

    frames = {
        "EVENTO": pd.DataFrame([{
            "ID Evento":             ev.get("id_evento", "N/D"),
            "Data / Ora":            ev.get("Data_Ora", "N/D"),
            "Tipo Evento":           ev.get("Tipo_Evento", "N/D"),
            "Gravità":               ev.get("Gravita_Stimata", "N/D"),
            "Situazione Pre-Evento": ev.get("Prima_dell_evento", "N/D"),
            "Trigger Principale":    ev.get("Trigger_Evento_Principale", "N/D"),
            "Dinamica (Sintesi)":    ev.get("Dinamica_Sintesi", "N/D"),
            }]),
        "LUOGHI & MEZZI": pd.DataFrame([{
            "Reparto":               lm.get("Reparto", "N/D"),
            "Area Coinvolta":        lm.get("Area_Coinvolta", "N/D"),
            "Macchinario":           lm.get("Macchinario_Coinvolto", "N/D"),
            "Codice Macchinario":    lm.get("Codice_Macchinario", "N/D"),
            "Condizioni Ambientali": lm.get("Condizioni_Ambientali", "N/D"),
        }]),
        "ANALISI CAUSE": pd.DataFrame([{
            "Causa Tecnica":              ac.get("Causa_Tecnica", "N/D"),
            "Fattore Umano":              ac.get("Fattore_Umano", "N/D"),
            "Fattore Ambientale":         ac.get("Fattore_Ambientale", "N/D"),
            "Causa Radice (Root Cause)":  ac.get("Causa_Radice", "N/D"),
        }]),
        "CONSEGUENZE": pd.DataFrame([{
            "Conseguenza Fisica":          co.get("Conseguenza_Fisica_Immediata", "N/D"),
            "Parte del Corpo Lesa":        co.get("Parte_Corpo_Lesa", "N/D"),
            "Giorni Inabilità (Stimati)":  co.get("Giorni_Inabilita_Stimati", "N/D"),
            "Conseguenza Materiale":       co.get("Conseguenza_Materiale_Aziendale", "N/D"),
            "Fermo Produzione (ore)":      co.get("Fermo_Produzione_Ore", "N/D"),
        }]),
        "AZIONI CORRETTIVE": pd.DataFrame([{
            "Procedura Violata":                  az.get("Procedura_Corretta_Violata", "N/D"),
            "DPI Mancanti/Inadeguati":            az.get("DPI_Mancanti_o_Inadeguati", "N/D"),
            "Azione Correttiva Immediata":        az.get("Azione_Correttiva_Immediata", "N/D"),
            "Azione Correttiva da Implementare":  az.get("Azione_Correttiva_da_Implementare", "N/D"),
            "Responsabile":                       az.get("Responsabile_Azione", "N/D"),
            "Scadenza":                           az.get("Scadenza_Azione", "N/D"),
        }]),
        "PERSONE COINVOLTE": (
            pd.DataFrame(pc).rename(columns=lambda c: c.replace("_", " "))
            if pc else
            pd.DataFrame(columns=["Nome", "Ruolo Aziendale", "Coinvolgimento",
                                   "Danno Riportato", "Affidabilita Testimonianza", "Note Testimonianza"])
        ),
    }
    return frames


# Larghezze colonne per foglio (in caratteri)
_LARGHEZZE = {
    "EVENTO":            [15, 18, 18, 12, 50, 50, 60, 50],
    "LUOGHI & MEZZI":    [25, 25, 30, 20, 35],
    "ANALISI CAUSE":     [45, 45, 35, 55],
    "CONSEGUENZE":       [40, 25, 22, 45, 22],
    "AZIONI CORRETTIVE": [45, 35, 45, 55, 25, 18],
    "PERSONE COINVOLTE": [20, 22, 18, 35, 22, 45],
}

print("✅ Utility Excel pronte.")


✅ Utility Excel pronte.


In [9]:
# =====================================================================
# CELLA 9: SALVATAGGIO EXCEL — Modalità A (un file per audio)
#per ogni risultato della Modalità A crea un file .xlsx separato
# =====================================================================

def salva_excel_singolo(risultato):
    """Salva un Excel per un singolo risultato (da analisi_singoli).
    risultato = {"percorso": ..., "nome": ..., "dati": {...}}
    """
    nome_base = os.path.splitext(risultato["nome"])[0]
    # Pulizia nome file da caratteri non validi
    nome_safe = re.sub(r'[<>:"/\\|?*]', '_', nome_base)
    file_output = os.path.join(OUTPUT_DIR, f"report_singolo_{nome_safe}.xlsx")

    frames = _dati_a_dataframes(risultato["dati"])

    with pd.ExcelWriter(file_output, engine="openpyxl") as writer:
        for sheet_name, df in frames.items():
            df.to_excel(writer, sheet_name=sheet_name, index=False)
        wb = writer.book
        for name in wb.sheetnames:
            _applica_stile(wb[name], _LARGHEZZE.get(name))

    print(f"   💾 Salvato: '{file_output}'")
    return file_output


def salva_excel_tutti_singoli(risultati):
    """Salva un file Excel per ciascun risultato della Modalità A."""
    print(f"\n📊 Modalità A — salvataggio {len(risultati)} file Excel individuali...")
    file_salvati = []
    for r in risultati:
        file_salvati.append(salva_excel_singolo(r))
    print(f"✅ {len(file_salvati)} file Excel salvati nella cartella '{OUTPUT_DIR}'.")
    return file_salvati

print("✅ Funzione salvataggio Modalità A pronta.")

✅ Funzione salvataggio Modalità A pronta.


In [10]:
# =====================================================================
# CELLA 10: SALVATAGGIO EXCEL — Modalità B (file unico multi-fonte)
#salva il report unificato della Modalità B in un singolo file .xlsx
# =====================================================================

def salva_excel_multi(dati, file_output=None):
    """Salva un Excel unificato dal risultato della Modalità B."""
    if file_output is None:
        file_output = os.path.join(OUTPUT_DIR, "report_multiaudio.xlsx")

    print(f"\n📊 Modalità B — salvataggio report multi-audio: '{file_output}'...")
    frames = _dati_a_dataframes(dati)

    with pd.ExcelWriter(file_output, engine="openpyxl") as writer:
        for sheet_name, df in frames.items():
            df.to_excel(writer, sheet_name=sheet_name, index=False)
        wb = writer.book
        for name in wb.sheetnames:
            _applica_stile(wb[name], _LARGHEZZE.get(name))

    print(f"✅ Report multi-audio salvato: '{file_output}'")
    return file_output

print("✅ Funzione salvataggio Modalità B pronta.")

✅ Funzione salvataggio Modalità B pronta.


In [11]:
# =====================================================================
# CELLA 11: ANTEPRIMA IN JUPYTER
# =====================================================================

# stampa un titolo Markdown e il DataFrame dell'Evento per ciascun audio, permettendo un controllo rapido dell'estrazione singola
def mostra_anteprima_singoli(risultati):
    """Mostra una tabella EVENTO per ciascun audio analizzato singolarmente."""
    display(Markdown("## 📋 MODALITÀ A — Anteprima per audio"))
    for r in risultati:
        display(Markdown(f"### 🎙️ Audio: `{r['nome']}`"))
        display(pd.DataFrame([r["dati"].get("Evento", {})]))


def mostra_anteprima_multi(dati):
    """Mostra l'anteprima del report multi-fonte."""
    display(Markdown("## 📋 MODALITÀ B — Anteprima report multi-audio"))
    display(Markdown("### Evento (sintesi incrociata)"))
    display(pd.DataFrame([dati.get("Evento", {})]))

# la funzione più utile per l'analisi: affianca su una stessa tabella le informazioni estratte da ciascun audio singolo (Modalità A) e il risultato consolidato (Modalità B), su colonne
def mostra_confronto(risultati_singoli, dati_multi):
    """Mostra una tabella di confronto affiancata delle sezioni chiave tra le due modalità."""
    display(Markdown("## 🔍 CONFRONTO — Dinamica Sintesi per modalità"))
    righe = []
    for r in risultati_singoli:
        righe.append({
            "Fonte": r["nome"],
            "Modalità": "A — Singolo",
            "Tipo Evento": r["dati"].get("Evento", {}).get("Tipo_Evento", "N/D"),
            "Gravità": r["dati"].get("Evento", {}).get("Gravita_Stimata", "N/D"),
            "Dinamica (Sintesi)": r["dati"].get("Evento", {}).get("Dinamica_Sintesi", "N/D"),
            "Causa Radice": r["dati"].get("Analisi_Cause", {}).get("Causa_Radice", "N/D"),
        })
    righe.append({
        "Fonte": f"{len(risultati_singoli)} audio combinati",
        "Modalità": "B — Multi-fonte",
        "Tipo Evento": dati_multi.get("Evento", {}).get("Tipo_Evento", "N/D"),
        "Gravità": dati_multi.get("Evento", {}).get("Gravita_Stimata", "N/D"),
        "Dinamica (Sintesi)": dati_multi.get("Evento", {}).get("Dinamica_Sintesi", "N/D"),
        "Causa Radice": dati_multi.get("Analisi_Cause", {}).get("Causa_Radice", "N/D"),
    })
    df_confronto = pd.DataFrame(righe)
    display(df_confronto)
    return df_confronto

print("✅ Funzioni anteprima pronte.")

✅ Funzioni anteprima pronte.


In [12]:
# =====================================================================
# CELLA 12: ▶️ ESEGUI — MODALITÀ A (analisi singola per ogni audio)
# =====================================================================

mancanti = [f for f in FILE_AUDIO if not os.path.exists(f)]
if mancanti:
    print(f"❌ File mancanti: {mancanti}")
else:
    # --- ANALISI ---
    risultati_singoli = analizza_singoli(FILE_AUDIO)

    # --- ANTEPRIMA ---
    mostra_anteprima_singoli(risultati_singoli)

    # --- SALVATAGGIO EXCEL ---
    file_singoli = salva_excel_tutti_singoli(risultati_singoli)

    print(f"\n🎉 Modalità A completata! {len(file_singoli)} file Excel in '{OUTPUT_DIR}/'")


🎙️ [1/3] Analisi singola: 'Elena RSPP.mp3'
   ⬆️  Upload: Elena RSPP.mp3
   ⏳ Attendo elaborazione ✅
   🧠 Estrazione dati...
   ✅ Completato.

🎙️ [2/3] Analisi singola: 'Marco Capolinea.mp3'
   ⬆️  Upload: Marco Capolinea.mp3
   ⏳ Attendo elaborazione ✅
   🧠 Estrazione dati...
   ✅ Completato.

🎙️ [3/3] Analisi singola: 'Martina Operatrice.mp3'
   ⬆️  Upload: Martina Operatrice.mp3
   ⏳ Attendo elaborazione ✅
   🧠 Estrazione dati...
   ✅ Completato.

✅ Modalità A completata: 3 audio analizzati singolarmente.


## 📋 MODALITÀ A — Anteprima per audio

### 🎙️ Audio: `Elena RSPP.mp3`

,id_evento,Data_Ora,Tipo_Evento,Gravita_Stimata,Prima_dell_evento,Trigger_Evento_Principale,Dinamica_Sintesi
0,N/D,N/D,Incidente,Bassa,Intervento manuale su una macchina in funzione,Introduzione della mano dell'operatore in una ...,Un operatore ha introdotto la mano in una zona...


### 🎙️ Audio: `Marco Capolinea.mp3`

,id_evento,Data_Ora,Tipo_Evento,Gravita_Stimata,Prima_dell_evento,Trigger_Evento_Principale,Dinamica_Sintesi
0,N/D,N/D 10:15,Incidente,Bassa,La linea di produzione (Linea 3) presentava di...,Un operatore è intervenuto manualmente per ria...,Durante un intervento manuale di riallineament...


### 🎙️ Audio: `Martina Operatrice.mp3`

,id_evento,Data_Ora,Tipo_Evento,Gravita_Stimata,Prima_dell_evento,Trigger_Evento_Principale,Dinamica_Sintesi
0,N/D,N/D,Incidente,Bassa,La termosaldatrice si bloccava di continuo sin...,Tentativo manuale dell'operatrice di riallinea...,"L'operatrice, sotto pressione per la produzion..."



📊 Modalità A — salvataggio 3 file Excel individuali...
   💾 Salvato: 'output_report\report_singolo_Elena RSPP.xlsx'
   💾 Salvato: 'output_report\report_singolo_Marco Capolinea.xlsx'
   💾 Salvato: 'output_report\report_singolo_Martina Operatrice.xlsx'
✅ 3 file Excel salvati nella cartella 'output_report'.

🎉 Modalità A completata! 3 file Excel in 'output_report/'


In [13]:
# =====================================================================
# CELLA 13: ▶️ ESEGUI — MODALITÀ B (fusione dei risultati Modalità A)
# ⚠️  Esegui PRIMA la Cella 13 (Modalità A)
# =====================================================================

try:
    # Verifichiamo che i dati della Modalità A siano presenti
    risultati_singoli
except NameError:
    print("❌ Errore: 'risultati_singoli' non trovato. Esegui la Cella 13 prima!")
else:
    # 1. Eseguiamo la fusione intelligente tramite Gemini
    dati_multi = fondi_risultati_singoli(risultati_singoli)

    # 2. Anteprima a video del risultato unificato
    mostra_anteprima_multi(dati_multi)

    # 3. Salvataggio definitivo nel file Excel unificato
    file_multi = salva_excel_multi(dati_multi)

    print(f"\n🎉 Modalità B completata con successo!")
    print(f"📊 Report finale unificato creato: '{file_multi}'")


🧠 Avvio fusione intelligente di 3 fonti con Gemini...


## 📋 MODALITÀ B — Anteprima report multi-audio

### Evento (sintesi incrociata)

,id_evento,Data_Ora,Tipo_Evento,Gravita_Stimata,Prima_dell_evento,Trigger_Evento_Principale,Dinamica_Sintesi
0,N/D,N/D 10:15,Incidente,Bassa,La linea di produzione (Linea 3) presentava di...,Tentativo manuale dell'operatrice di riallinea...,"Un'operatrice, sotto pressione produttiva e pe..."



📊 Modalità B — salvataggio report multi-audio: 'output_report\report_multiaudio.xlsx'...
✅ Report multi-audio salvato: 'output_report\report_multiaudio.xlsx'

🎉 Modalità B completata con successo!
📊 Report finale unificato creato: 'output_report\report_multiaudio.xlsx'


---
## 🔬 MODALITÀ C — Analisi Qualità Multi-Estrazione

Le celle seguenti implementano la Modalità C.
Esegui nell'ordine:
1. **Cella 15** — Configurazione N estrazioni
2. **Cella 16** — Trascrizione audio (opzionale, per metriche CER/WER/DTA/EPR)
3. **Cella 17** — Funzioni Modalità C (logica estrazione + metriche)
4. **Cella 18** — Salvataggio Excel Modalità C
5. **Cella 19** — ▶️ ESEGUI Modalità C


In [14]:
# =====================================================================
# CELLA 15: CONFIGURAZIONE MODALITÀ C
# =====================================================================

# ---------------------------------------------------------------
# 🔢 Numero di estrazioni per ciascun audio
#    Aumenta per maggiore robustezza statistica (min consigliato: 3)
# ---------------------------------------------------------------
N_ESTRAZIONI = 3

# ---------------------------------------------------------------
# 📚 TERMINI TECNICI HSE — usati per Domain Term Accuracy (DTA)
#    Aggiungi o rimuovi termini in base al tuo dominio
# ---------------------------------------------------------------
TERMINI_TECNICI_HSE = [
    "pressa", "tornio", "valvola", "nastro", "DPI", "RSPP",
    "near miss", "root cause", "incidente", "infortunio",
    "reparto", "macchinario", "fermo produzione", "causa radice",
    "azione correttiva", "trigger", "gravità", "dinamica"
]

# ---------------------------------------------------------------
# 📝 GROUND TRUTH TRASCRIZIONI (opzionale — per CER/WER/EPR)
#    Se hai la trascrizione manuale corretta di ogni audio,
#    inseriscila qui come stringa. Lascia "" per saltare.
#    La chiave deve corrispondere esattamente al nome del file audio.
# ---------------------------------------------------------------
GROUND_TRUTH_TRASCRIZIONI = {
    "Elena RSPP.mp3":         "",   # es. "Il giorno 15 marzo alle ore..."
    "Marco Capolinea.mp3":    "",
    "Martina Operatrice.mp3": "",
}

print(f"✅ Configurazione Modalità C:")
print(f"   N estrazioni per audio: {N_ESTRAZIONI}")
print(f"   Termini tecnici HSE configurati: {len(TERMINI_TECNICI_HSE)}")
gt_presenti = sum(1 for v in GROUND_TRUTH_TRASCRIZIONI.values() if v.strip())
print(f"   Ground truth trascrizioni fornite: {gt_presenti}/{len(GROUND_TRUTH_TRASCRIZIONI)}")


✅ Configurazione Modalità C:
   N estrazioni per audio: 3
   Termini tecnici HSE configurati: 18
   Ground truth trascrizioni fornite: 0/3


In [15]:
# =====================================================================
# CELLA 16: TRASCRIZIONE AUDIO (opzionale)
# Chiede a Gemini la trascrizione grezza di ogni audio N_ESTRAZIONI volte,
# cosi da avere una trascrizione per ogni run (usata nel foglio TRASCRIZIONI
# e per le metriche CER/WER/DTA/EPR).
# Ometti questa cella se non ti servono le metriche di trascrizione:
# le colonne corrispondenti mostreranno "N/D".
# =====================================================================

def trascrivi_audio_singolo(percorso):
    """Una singola chiamata di trascrizione su un audio."""
    file_gemini = _upload_e_attendi([percorso])
    prompt_trascrizione = (
        "Trascrivi fedelmente tutto il parlato contenuto in questo audio.\n"
        "Riporta SOLO il testo trascritto, senza aggiunte, etichette del parlante\n"
        "o punteggiatura inventata. Non correggere errori grammaticali o di pronuncia:\n"
        "trascrivi esattamente quello che senti."
    )
    model = genai.GenerativeModel("gemini-2.5-flash")
    risposta = model.generate_content([prompt_trascrizione] + file_gemini)
    _cleanup(file_gemini)
    return risposta.text.strip()


def trascrivi_tutti_multirun(lista_percorsi, n_estrazioni=N_ESTRAZIONI):
    """
    Per ogni audio esegue N_ESTRAZIONI trascrizioni indipendenti.
    Restituisce:
        {
          "Elena RSPP.mp3": ["trascrizione run1", "trascrizione run2", ...],
          ...
        }
    """
    risultato = {}
    for percorso in lista_percorsi:
        nome = os.path.basename(percorso)
        if not os.path.exists(percorso):
            print(f"   File non trovato: {nome}")
            risultato[nome] = [""] * n_estrazioni
            continue
        print(f"\n  Trascrizione multi-run: {nome}")
        runs_trasc = []
        for run in range(1, n_estrazioni + 1):
            print(f"    Run {run}/{n_estrazioni}...", end=" ")
            t = trascrivi_audio_singolo(percorso)
            runs_trasc.append(t)
            print(f"OK ({len(t)} caratteri)")
        risultato[nome] = runs_trasc
    return risultato


# ESECUZIONE
mancanti_c = [f for f in FILE_AUDIO if not os.path.exists(f)]
if mancanti_c:
    print(f"File mancanti: {mancanti_c}")
else:
    print("Avvio trascrizione multi-run per Modalita C...\n")
    trascrizioni_c = trascrivi_tutti_multirun(FILE_AUDIO, N_ESTRAZIONI)
    print("\nTrascrizioni completate. Variabile 'trascrizioni_c' disponibile.")
    for nome, runs_t in trascrizioni_c.items():
        print(f"  {nome}: {len(runs_t)} trascrizioni")


Avvio trascrizione multi-run per Modalita C...


  Trascrizione multi-run: Elena RSPP.mp3
    Run 1/3...    ⬆️  Upload: Elena RSPP.mp3
   ⏳ Attendo elaborazione ✅
OK (1316 caratteri)
    Run 2/3...    ⬆️  Upload: Elena RSPP.mp3
   ⏳ Attendo elaborazione ✅
OK (1329 caratteri)
    Run 3/3...    ⬆️  Upload: Elena RSPP.mp3
   ⏳ Attendo elaborazione ✅
OK (1330 caratteri)

  Trascrizione multi-run: Marco Capolinea.mp3
    Run 1/3...    ⬆️  Upload: Marco Capolinea.mp3
   ⏳ Attendo elaborazione ✅
OK (1168 caratteri)
    Run 2/3...    ⬆️  Upload: Marco Capolinea.mp3
   ⏳ Attendo elaborazione ✅
OK (1151 caratteri)
    Run 3/3...    ⬆️  Upload: Marco Capolinea.mp3
   ⏳ Attendo elaborazione ✅
OK (1178 caratteri)

  Trascrizione multi-run: Martina Operatrice.mp3
    Run 1/3...    ⬆️  Upload: Martina Operatrice.mp3
   ⏳ Attendo elaborazione ✅
OK (1224 caratteri)
    Run 2/3...    ⬆️  Upload: Martina Operatrice.mp3
   ⏳ Attendo elaborazione ✅
OK (1252 caratteri)
    Run 3/3...    ⬆️  Upload: Martina 

In [16]:
# =====================================================================
# CELLA 17: FUNZIONI MODALITA C — estrazione multi-run + metriche
# =====================================================================

# ──────────────────────────────────────────────────────────────────
# SEZIONE 1: ESTRAZIONE MULTI-RUN
# ──────────────────────────────────────────────────────────────────

def analizza_modalita_c(percorso, n_estrazioni=N_ESTRAZIONI):
    """
    Analizza un singolo audio per N_ESTRAZIONI volte indipendenti.
    Ritorna lista di dict: [{"estrazione": i, "nome": ..., "dati": {...}}, ...]
    """
    nome = os.path.basename(percorso)
    print(f"\nModalita C — '{nome}' ({n_estrazioni} estrazioni)")
    runs = []
    for run in range(1, n_estrazioni + 1):
        print(f"   Estrazione {run}/{n_estrazioni}...")
        file_caricati = _upload_e_attendi([percorso])
        prompt = build_prompt_singolo(nome, run)
        try:
            dati = _chiedi_gemini(prompt, file_caricati)
        except Exception as e:
            print(f"   Errore estrazione {run}: {e}")
            dati = {}
        _cleanup(file_caricati)
        runs.append({"estrazione": run, "nome": nome, "dati": dati})
        print(f"   Estrazione {run} completata.")
    return runs


# ──────────────────────────────────────────────────────────────────
# SEZIONE 2: GROUND TRUTH PROXY
# ──────────────────────────────────────────────────────────────────

def _valore_str(dati, sezione, campo):
    """Estrae un valore stringa da dati JSON annidato."""
    if sezione == "Persone_Coinvolte":
        persone = dati.get(sezione, [])
        if isinstance(persone, list) and len(persone) > 0:
            v = persone[0].get(campo, "N/D") if isinstance(persone[0], dict) else "N/D"
        else:
            v = "N/D"
    else:
        v = dati.get(sezione, {}).get(campo, "N/D")
    if isinstance(v, list):
        return " ".join(str(x) for x in v)
    return str(v) if v else ""


CAMPI_CATEGORICI = [
    ("Evento",            "Tipo_Evento"),
    ("Evento",            "Gravita_Stimata"),
    ("Persone_Coinvolte", "Coinvolgimento"),
]

CAMPI_TESTUALI = [
    ("Evento",            "Prima_dell_evento"),
    ("Evento",            "Trigger_Evento_Principale"),
    ("Evento",            "Dinamica_Sintesi"),
    ("Luoghi_e_Mezzi",    "Reparto"),
    ("Luoghi_e_Mezzi",    "Macchinario_Coinvolto"),
    ("Luoghi_e_Mezzi",    "Condizioni_Ambientali"),
    ("Analisi_Cause",     "Causa_Tecnica"),
    ("Analisi_Cause",     "Fattore_Umano"),
    ("Analisi_Cause",     "Causa_Radice"),
    ("Conseguenze",       "Conseguenza_Fisica_Immediata"),
    ("Conseguenze",       "Parte_Corpo_Lesa"),
    ("Azioni_Correttive", "Azione_Correttiva_Immediata"),
    ("Azioni_Correttive", "Responsabile_Azione"),
]


def _ground_truth_proxy(runs):
    """Ground truth proxy: moda per categorici, stringa piu lunga per testuali."""
    gt = {}
    for sezione, campo in CAMPI_TESTUALI + CAMPI_CATEGORICI:
        valori = [_valore_str(r["dati"], sezione, campo) for r in runs if r["dati"]]
        valori_validi = [v for v in valori if v and v != "N/D"]
        if not valori_validi:
            gt[(sezione, campo)] = "N/D"
        elif (sezione, campo) in [(s, c) for s, c in CAMPI_CATEGORICI]:
            cnt = Counter(valori_validi)
            gt[(sezione, campo)] = cnt.most_common(1)[0][0]
        else:
            gt[(sezione, campo)] = max(valori_validi, key=len)
    return gt


# ──────────────────────────────────────────────────────────────────
# SEZIONE 3: METRICHE REPORT
# ──────────────────────────────────────────────────────────────────

def _jaccard(a, b):
    """Similarita Jaccard tra due stringhe (token-based)."""
    if not a or not b:
        return 0.0
    sa = set(a.lower().split())
    sb = set(b.lower().split())
    inter = len(sa & sb)
    union = len(sa | sb)
    return inter / union if union else 0.0


def _precision_recall_f1(estratto, riferimento, soglia=0.45):
    """Precision, Recall, F1 confrontando campi testuali con ground truth proxy."""
    tp = fp = fn = 0
    for sezione, campo in CAMPI_TESTUALI:
        val_est = _valore_str(estratto, sezione, campo)
        val_gt  = riferimento.get((sezione, campo), "N/D")
        est_presente = val_est not in ("", "N/D")
        gt_presente  = val_gt  not in ("", "N/D")
        if est_presente and gt_presente:
            if _jaccard(val_est, val_gt) >= soglia:
                tp += 1
            else:
                fp += 1
        elif est_presente and not gt_presente:
            fp += 1
        elif not est_presente and gt_presente:
            fn += 1
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)
    return round(precision, 3), round(recall, 3), round(f1, 3)


def _cohen_kappa_runs(runs, sezione, campo):
    """Cohen's Kappa medio tra tutte le coppie di run su campi categorici."""
    valori = [_valore_str(r["dati"], sezione, campo) for r in runs]
    if len(set(valori)) == 1:
        return 1.0
    kappas = []
    for i, j in itertools.combinations(range(len(runs)), 2):
        if valori[i] == valori[j]:
            kappas.append(1.0)
        else:
            labels_i = [_valore_str(runs[i]["dati"], s, c) for s, c in CAMPI_CATEGORICI]
            labels_j = [_valore_str(runs[j]["dati"], s, c) for s, c in CAMPI_CATEGORICI]
            try:
                k = cohen_kappa_score(labels_i, labels_j)
                kappas.append(k)
                break
            except Exception:
                kappas.append(0.0)
                break
    return round(sum(kappas) / len(kappas), 3) if kappas else 0.0


def _consistenza_lessicale(runs, sezione, campo):
    """Jaccard medio tra tutte le coppie di run per un campo testuale."""
    valori = [_valore_str(r["dati"], sezione, campo) for r in runs]
    sims = [_jaccard(valori[i], valori[j])
            for i, j in itertools.combinations(range(len(valori)), 2)]
    return round(sum(sims) / len(sims), 3) if sims else 1.0


# ──────────────────────────────────────────────────────────────────
# SEZIONE 4: METRICHE TRASCRIZIONE
# ──────────────────────────────────────────────────────────────────

def _cer(ipotesi, riferimento):
    """Character Error Rate (Levenshtein su caratteri)."""
    if not riferimento:
        return None
    n, m = len(riferimento), len(ipotesi)
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev = dp[:]
        dp[0] = i
        for j in range(1, m + 1):
            dp[j] = (prev[j-1] if riferimento[i-1] == ipotesi[j-1]
                     else 1 + min(prev[j-1], prev[j], dp[j-1]))
    return round(dp[m] / n, 4)


def _wer(ipotesi, riferimento):
    """Word Error Rate (Levenshtein su parole)."""
    if not riferimento:
        return None
    ref_w, hyp_w = riferimento.split(), ipotesi.split()
    n, m = len(ref_w), len(hyp_w)
    if n == 0:
        return 0.0
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev = dp[:]
        dp[0] = i
        for j in range(1, m + 1):
            dp[j] = (prev[j-1] if ref_w[i-1] == hyp_w[j-1]
                     else 1 + min(prev[j-1], prev[j], dp[j-1]))
    return round(dp[m] / n, 4)


def _dta(ipotesi, termini):
    """Domain Term Accuracy: proporzione di termini HSE presenti nella trascrizione."""
    if not ipotesi:
        return None
    testo = ipotesi.lower()
    corretti = sum(1 for t in termini if t.lower() in testo)
    return round(corretti / len(termini), 4) if termini else None


def _epr(trascrizione, dati_estratti, soglia_jaccard=0.3):
    """Error Propagation Rate (euristica): campi JSON potenzialmente inquinati da errori."""
    if not trascrizione:
        return None
    parole_trasc = set(trascrizione.lower().split())
    campi_err = campi_tot = 0
    for sezione, campo in CAMPI_TESTUALI:
        val = _valore_str(dati_estratti, sezione, campo)
        if val and val != "N/D":
            campi_tot += 1
            parole_campo = set(val.lower().split())
            if parole_trasc and parole_campo:
                overlap = len(parole_campo & parole_trasc) / len(parole_campo)
                if overlap < soglia_jaccard:
                    campi_err += 1
    return round(campi_err / campi_tot, 4) if campi_tot > 0 else None


# ──────────────────────────────────────────────────────────────────
# SEZIONE 5: CONFRONTO TRASCRIZIONI — parole diverse
# ──────────────────────────────────────────────────────────────────

def _parole_diverse(testo, riferimento):
    """
    Dato un testo e un riferimento (la trascrizione di riferimento),
    restituisce una lista di tuple (parola, is_diversa: bool)
    dove is_diversa=True indica che la parola NON compare nel riferimento.
    Utile per colorare in rosso le parole difformi nel foglio Excel.
    """
    if not testo:
        return []
    parole_rif = set(riferimento.lower().split()) if riferimento else set()
    risultato = []
    for parola in testo.split():
        parola_pulita = parola.lower().strip(".,;:!?()\"'")
        is_diversa = parola_pulita not in parole_rif
        risultato.append((parola, is_diversa))
    return risultato


# ──────────────────────────────────────────────────────────────────
# SEZIONE 6: CALCOLO METRICHE COMPLETE
# ──────────────────────────────────────────────────────────────────

def calcola_metriche_c(runs, trascrizioni_runs=None, ground_truth_testo=None,
                        nome_audio=None):
    """
    Calcola tutte le metriche di qualita per i runs di un audio.
    trascrizioni_runs: lista di stringhe, una per run (da Cella 16).
    Restituisce lista di dict (una riga per run) + riga MEDIA finale.
    """
    gt_proxy = _ground_truth_proxy(runs)
    nome_audio = nome_audio or runs[0]["nome"]
    gt_testo   = ground_truth_testo or ""

    kappa_globale = _cohen_kappa_runs(runs, "Evento", "Tipo_Evento")
    consistenze = [
        _consistenza_lessicale(runs, s, c)
        for s, c in [
            ("Evento",        "Dinamica_Sintesi"),
            ("Analisi_Cause", "Causa_Radice"),
            ("Luoghi_e_Mezzi","Macchinario_Coinvolto"),
        ]
    ]
    consist_media = round(sum(consistenze) / len(consistenze), 3)

    righe = []
    for idx, r in enumerate(runs):
        prec, rec, f1 = _precision_recall_f1(r["dati"], gt_proxy)
        trasc_run = (trascrizioni_runs[idx]
                     if trascrizioni_runs and idx < len(trascrizioni_runs)
                     else None)

        cer_val = dta_val = wer_val = epr_val = "N/D (no ground truth)"
        if gt_testo.strip() and trasc_run:
            cer_val = _cer(trasc_run, gt_testo)
            wer_val = _wer(trasc_run, gt_testo)
            dta_val = _dta(trasc_run, TERMINI_TECNICI_HSE)
            epr_val = _epr(trasc_run, r["dati"])
        elif trasc_run:
            dta_val = _dta(trasc_run, TERMINI_TECNICI_HSE)
            epr_val = _epr(trasc_run, r["dati"])

        righe.append({
            "Audio":                   nome_audio,
            "Estrazione #":            r["estrazione"],
            "Cohen's Kappa (globale)": kappa_globale,
            "Precision":               prec,
            "Recall":                  rec,
            "F1 Score":                f1,
            "Consistenza Lessicale":   consist_media,
            "CER":                     cer_val,
            "WER":                     wer_val,
            "DTA":                     dta_val,
            "EPR":                     epr_val,
        })

    def media_col(col):
        vals = [r[col] for r in righe if isinstance(r[col], float)]
        return round(sum(vals)/len(vals), 3) if vals else "N/D"

    righe.append({
        "Audio":                   nome_audio,
        "Estrazione #":            "MEDIA",
        "Cohen's Kappa (globale)": media_col("Cohen's Kappa (globale)"),
        "Precision":               media_col("Precision"),
        "Recall":                  media_col("Recall"),
        "F1 Score":                media_col("F1 Score"),
        "Consistenza Lessicale":   media_col("Consistenza Lessicale"),
        "CER":                     media_col("CER") if isinstance(righe[0]["CER"], float) else "N/D",
        "WER":                     media_col("WER") if isinstance(righe[0]["WER"], float) else "N/D",
        "DTA":                     media_col("DTA") if isinstance(righe[0]["DTA"], float) else "N/D",
        "EPR":                     media_col("EPR") if isinstance(righe[0]["EPR"], float) else "N/D",
    })
    return righe


print("Funzioni Modalita C pronte.")
print(f"  Campi categorici monitorati: {len(CAMPI_CATEGORICI)}")
print(f"  Campi testuali monitorati:   {len(CAMPI_TESTUALI)}")


Funzioni Modalita C pronte.
  Campi categorici monitorati: 3
  Campi testuali monitorati:   13


In [19]:
# =====================================================================
# CELLA 18: SALVATAGGIO EXCEL MODALITA C
# Fogli prodotti per ogni audio:
#   EVENTO | LUOGHI & MEZZI | ANALISI CAUSE | CONSEGUENZE |
#   AZIONI CORRETTIVE | PERSONE COINVOLTE |
#   TRASCRIZIONI (richtext, parole diverse in rosso) |
#   ANALISI QUALITA (metriche con colori semaforo + legenda)
# =====================================================================
from openpyxl.cell.rich_text import CellRichText as RichText, TextBlock
from openpyxl.cell.text import InlineFont
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import re, os

# ── PALETTE SEMAFORO ──────────────────────────────────────────────
_C_VERDE   = "C6EFCE"
_C_GIALLO  = "FFEB9C"
_C_ARANCIO = "FDCDA0"
_C_ROSSO   = "FFC7CE"
_C_GRIGIO  = "F2F2F2"

def _fill(hex_color):
    return PatternFill(start_color=hex_color, end_color=hex_color, fill_type="solid")

def _side(style="thin", color="BFBFBF"):
    return Side(style=style, color=color)

def _border(style="thin", color="BFBFBF"):
    s = _side(style, color)
    return Border(left=s, right=s, top=s, bottom=s)

BORDER_THICK_TOP = Border(
    left=_side(), right=_side(), bottom=_side(),
    top=_side("medium", "1A5276")
)

# ── SOGLIE METRICHE ───────────────────────────────────────────────
# (ottimo, buono, scarso, verso, gruppo, descrizione)
# gruppo: "report" | "trascrizione"
_SOGLIE_METRICHE = {
    "Cohen's Kappa (globale)": (0.81, 0.61, 0.40, "alto",  "report",       "Accordo tra estrazioni"),
    "Precision":               (0.85, 0.70, 0.50, "alto",  "report",       "Accuratezza estrazione"),
    "Recall":                  (0.85, 0.70, 0.50, "alto",  "report",       "Copertura informazioni"),
    "F1 Score":                (0.85, 0.70, 0.50, "alto",  "report",       "Qualita bilanciata"),
    "Consistenza Lessicale":   (0.75, 0.55, 0.35, "alto",  "report",       "Stabilita lessicale tra run"),
    "CER":                     (0.05, 0.15, 0.30, "basso", "trascrizione", "Errori a livello carattere"),
    "WER":                     (0.10, 0.25, 0.40, "basso", "trascrizione", "Errori a livello parola"),
    "DTA":                     (0.90, 0.75, 0.50, "alto",  "trascrizione", "Termini HSE riconosciuti"),
    "EPR":                     (0.10, 0.25, 0.40, "basso", "trascrizione", "Propagazione errori"),
}

# Colori intestazione per i due gruppi di metriche
_FILL_HDR_REPORT     = PatternFill(start_color="1A5276", end_color="1A5276", fill_type="solid")
_FILL_HDR_TRASC      = PatternFill(start_color="1A6B5E", end_color="1A6B5E", fill_type="solid")
_FILL_HDR_BASE       = PatternFill(start_color="2C3E50", end_color="2C3E50", fill_type="solid")

QUALITA_HEADER_FILL  = _FILL_HDR_BASE
MEDIA_FILL           = PatternFill(start_color="EAF2FF", end_color="EAF2FF", fill_type="solid")
MEDIA_FONT           = Font(bold=True, size=10, name="Arial", color="1A3A6B")

_LARGHEZZE_C = {
    "EVENTO":            [14, 15, 18, 18, 12, 50, 50, 60],
    "LUOGHI & MEZZI":    [14, 25, 25, 30, 20, 35],
    "ANALISI CAUSE":     [14, 45, 45, 35, 55],
    "CONSEGUENZE":       [14, 40, 25, 22, 45, 22],
    "AZIONI CORRETTIVE": [14, 45, 35, 45, 55, 25, 18],
    "PERSONE COINVOLTE": [14, 20, 22, 18, 35, 22, 45],
    "TRASCRIZIONI":      [14, 130],
    # Audio(22) | Estr(12) | Kappa(16) | Prec(14) | Rec(14) | F1(14) |
    # Consist(18) | CER(14) | WER(14) | DTA(14) | EPR(14)
    "ANALISI QUALITA":   [22, 12, 16, 14, 14, 14, 18, 14, 14, 14, 14],
}


def _colore_metrica(nome_colonna, valore):
    """Colore semaforo per una cella metrica in base al valore."""
    if nome_colonna not in _SOGLIE_METRICHE:
        return None
    if not isinstance(valore, (int, float)):
        return _C_GRIGIO
    s_ott, s_buo, s_sca, verso = _SOGLIE_METRICHE[nome_colonna][:4]
    if verso == "alto":
        if valore >= s_ott:   return _C_VERDE
        elif valore >= s_buo: return _C_GIALLO
        elif valore >= s_sca: return _C_ARANCIO
        else:                 return _C_ROSSO
    else:
        if valore <= s_ott:   return _C_VERDE
        elif valore <= s_buo: return _C_GIALLO
        elif valore <= s_sca: return _C_ARANCIO
        else:                 return _C_ROSSO


def _normalizza_parola(p):
    """Rimuove punteggiatura da entrambi i lati e porta in minuscolo."""
    return p.lower().strip(".,;:!?()[]{}\"\'«»…–—")


# ── FOGLIO TRASCRIZIONI ───────────────────────────────────────────

def _aggiungi_foglio_trascrizioni(wb, runs, trascrizioni_runs):
    """
    Crea il foglio TRASCRIZIONI con una riga per run.

    Logica confronto:
    - Costruisce il vocabolario UNIONE di tutte le trascrizioni (normalizzate)
    - Costruisce il vocabolario INTERSEZIONE (parole presenti in TUTTI i run)
    - Una parola è evidenziata in ROSSO se NON compare in almeno un altro run
      (cioè non e' nell'intersezione), cosi da segnalare differenze reali
      tra trascrizioni indipendentemente dalla punteggiatura.
    """
    ws = wb.create_sheet("TRASCRIZIONI")

    ws.column_dimensions["A"].width = _LARGHEZZE_C["TRASCRIZIONI"][0]
    ws.column_dimensions["B"].width = _LARGHEZZE_C["TRASCRIZIONI"][1]
    ws.row_dimensions[1].height = 32

    for col, val in [(1, "Estrazione #"),
                     (2, "Testo Trascritto  (parole in ROSSO = compaiono solo in questa estrazione)")]:
        c = ws.cell(row=1, column=col, value=val)
        c.font      = Font(bold=True, size=11, color="FFFFFF", name="Arial")
        c.fill      = QUALITA_HEADER_FILL
        c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        c.border    = _border()

    trascrizioni_valide = [t for t in (trascrizioni_runs or []) if t and t.strip()]

    # Vocabolari normalizzati per ogni run
    voc_runs = []
    for t in trascrizioni_valide:
        voc_runs.append({_normalizza_parola(p) for p in t.split()})

    # Intersezione = parole comuni a TUTTI i run
    if voc_runs:
        parole_comuni = voc_runs[0].copy()
        for v in voc_runs[1:]:
            parole_comuni &= v
    else:
        parole_comuni = set()

    font_nero  = InlineFont(rFont="Arial", sz=20, color="000000")
    font_rosso = InlineFont(rFont="Arial", sz=20, b=True, color="CC0000")

    for row_idx, r in enumerate(runs, start=2):
        trasc = (trascrizioni_runs[r["estrazione"] - 1]
                 if trascrizioni_runs and r["estrazione"] - 1 < len(trascrizioni_runs)
                 else None)

        # altezza riga automatica: ~15pt per ogni 130 caratteri stimati per riga
        if trasc:
            n_righe_stimate = max(1, len(trasc) // 100)
            ws.row_dimensions[row_idx].height = min(20 + n_righe_stimate * 15, 300)
        else:
            ws.row_dimensions[row_idx].height = 22

        cella_idx = ws.cell(row=row_idx, column=1, value=r["estrazione"])
        cella_idx.font      = Font(size=11, name="Arial", bold=True)
        cella_idx.alignment = Alignment(horizontal="center", vertical="top")
        cella_idx.border    = _border()
        if row_idx % 2 == 0:
            cella_idx.fill = _fill("EBF5FB")

        cella_txt = ws.cell(row=row_idx, column=2)
        cella_txt.border    = _border()
        cella_txt.alignment = Alignment(wrap_text=True, vertical="top")

        if not trasc or not trasc.strip():
            cella_txt.value = "N/D — esegui la Cella 16 per ottenere le trascrizioni"
            cella_txt.font  = Font(size=10, name="Arial", italic=True, color="888888")
            if row_idx % 2 == 0:
                cella_txt.fill = _fill("EBF5FB")
            continue

        try:
            blocchi = []
            parole = trasc.split()
            for i_p, parola in enumerate(parole):
                parola_norm = _normalizza_parola(parola)
                # Rosso se la parola NON e' nell'intersezione (differisce tra run)
                is_diversa = bool(parole_comuni) and (parola_norm not in parole_comuni)
                spazio = " " if i_p < len(parole) - 1 else ""
                blocchi.append(TextBlock(
                    font_rosso if is_diversa else font_nero,
                    parola + spazio
                ))
            cella_txt.value = RichText(blocchi)
        except Exception:
            cella_txt.value = trasc
            cella_txt.font  = Font(size=10, name="Arial")

        if row_idx % 2 == 0:
            cella_txt.fill = _fill("EBF5FB")

    ws.freeze_panes = "A2"
    return ws


# ── FOGLIO ANALISI QUALITA ────────────────────────────────────────

def _applica_stile_qualita(ws, larghezze=None):
    """
    Layout rivisitato:
    - Riga 1: intestazioni con due fasce cromatiche
      (blu scuro = Audio/Estrazione + parametri report |
       verde petrolio = parametri trascrizione)
    - Righe dati: colori semaforo sulle metriche, testo centrato e leggibile
    - Riga MEDIA: sfondo azzurro chiaro + bordo superiore blu spesso
    - Legenda: tabella compatta con campione colore + range per metrica
    """
    intestazioni = [cell.value for cell in ws[1]]

    # --- intestazione con doppia fascia cromatica ---
    COLONNE_TRASC = {"CER", "WER", "DTA", "EPR"}
    for col_idx, cell in enumerate(ws[1]):
        nome = intestazioni[col_idx] or ""
        cell.font      = Font(bold=True, size=10, color="FFFFFF", name="Arial")
        cell.alignment = Alignment(horizontal="center", vertical="center",
                                   wrap_text=True)
        cell.border    = _border("thin", "FFFFFF")
        if nome in COLONNE_TRASC:
            cell.fill = _FILL_HDR_TRASC
        else:
            cell.fill = _FILL_HDR_REPORT
    ws.row_dimensions[1].height = 40

    # --- righe dati ---
    n_righe_dati = ws.max_row
    for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):
        is_media = str(row[1].value).upper() == "MEDIA"
        for col_idx, cell in enumerate(row):
            col_name = intestazioni[col_idx] if col_idx < len(intestazioni) else ""
            colore   = _colore_metrica(col_name, cell.value)

            cell.alignment = Alignment(horizontal="center", vertical="center",
                                       wrap_text=True)
            cell.font   = Font(bold=is_media, size=10, name="Arial",
                               color="1A3A6B" if is_media else "000000")

            if is_media:
                cell.border = BORDER_THICK_TOP
                if colore and colore != _C_GRIGIO:
                    cell.fill = _fill(colore)
                    # testo più scuro per leggibilità sul colore
                    cell.font = Font(bold=True, size=10, name="Arial", color="000000")
                else:
                    cell.fill = MEDIA_FILL
                    cell.font = MEDIA_FONT
            else:
                cell.border = _border()
                if colore:
                    cell.fill = _fill(colore)
                elif row_idx % 2 == 0:
                    cell.fill = _fill("F0F4F8")

        ws.row_dimensions[row_idx].height = 22

    # --- larghezze colonne ---
    for i, col in enumerate(ws.columns):
        letter = get_column_letter(col[0].column)
        if larghezze and i < len(larghezze):
            ws.column_dimensions[letter].width = larghezze[i]
        else:
            max_len = max((len(str(c.value or "")) for c in col), default=10)
            ws.column_dimensions[letter].width = min(max(max_len * 0.9, 12), 40)

    ws.freeze_panes = "B2"   # congela colonna Audio + riga intestazione

    # ── LEGENDA ──────────────────────────────────────────────────
    riga_leg = ws.max_row + 3

    # Titolo legenda
    t = ws.cell(row=riga_leg, column=1,
                value="LEGENDA — Interpretazione colori e soglie")
    t.font = Font(bold=True, size=11, name="Arial", color="1A5276")
    ws.merge_cells(start_row=riga_leg, start_column=1,
                   end_row=riga_leg, end_column=8)
    ws.row_dimensions[riga_leg].height = 22
    riga_leg += 1

    # Sottotitoli gruppo
    for testo, col_start, col_end, fill_hdr in [
        ("Parametri Report",       1, 4, _FILL_HDR_REPORT),
        ("Parametri Trascrizione", 5, 8, _FILL_HDR_TRASC),
    ]:
        c = ws.cell(row=riga_leg, column=col_start, value=testo)
        c.font  = Font(bold=True, size=9, color="FFFFFF", name="Arial")
        c.fill  = fill_hdr
        c.alignment = Alignment(horizontal="center", vertical="center")
        c.border = _border("thin", "FFFFFF")
        ws.merge_cells(start_row=riga_leg, start_column=col_start,
                       end_row=riga_leg, end_column=col_end)
    ws.row_dimensions[riga_leg].height = 18
    riga_leg += 1

    # Intestazione colonne legenda
    hdrs_leg = ["Metrica", "Ottimo", "Buono", "Scarso / Critico", "Metrica",
                "Ottimo", "Buono", "Scarso / Critico"]
    hdr_fill_leg = PatternFill(start_color="2C3E50", end_color="2C3E50", fill_type="solid")
    for col_i, h in enumerate(hdrs_leg, start=1):
        c = ws.cell(row=riga_leg, column=col_i, value=h)
        c.font      = Font(bold=True, size=8, color="FFFFFF", name="Arial")
        c.fill      = hdr_fill_leg
        c.alignment = Alignment(horizontal="center", vertical="center")
        c.border    = _border()
    ws.row_dimensions[riga_leg].height = 16
    riga_leg += 1

    # Righe metriche affiancate (report a sinistra, trascrizione a destra)
    metriche_report = [(k, v) for k, v in _SOGLIE_METRICHE.items()
                       if v[4] == "report"]
    metriche_trasc  = [(k, v) for k, v in _SOGLIE_METRICHE.items()
                       if v[4] == "trascrizione"]
    n_rows_leg = max(len(metriche_report), len(metriche_trasc))

    for i in range(n_rows_leg):
        ws.row_dimensions[riga_leg + i].height = 16
        for col_offset, metriche_gruppo in [(0, metriche_report), (4, metriche_trasc)]:
            if i >= len(metriche_gruppo):
                continue
            metrica, (s_ott, s_buo, s_sca, verso, _, desc) = metriche_gruppo[i]
            if verso == "alto":
                r_ott = f">= {s_ott}"
                r_buo = f"{s_buo}–{s_ott}"
                r_sca = f"< {s_buo} / < {s_sca}"
            else:
                r_ott = f"<= {s_ott}"
                r_buo = f"{s_ott}–{s_buo}"
                r_sca = f"> {s_buo} / > {s_sca}"

            riga_dati = [
                (metrica, None),
                (r_ott,   _C_VERDE),
                (r_buo,   _C_GIALLO),
                (r_sca,   _C_ARANCIO),
            ]
            for col_j, (val, col_hex) in enumerate(riga_dati):
                c = ws.cell(row=riga_leg + i, column=col_offset + col_j + 1, value=val)
                c.font      = Font(size=8, name="Arial",
                                   bold=(col_j == 0),
                                   italic=(col_j == 0))
                c.alignment = Alignment(horizontal="center", vertical="center",
                                        wrap_text=True)
                c.border    = _border()
                if col_hex:
                    c.fill = _fill(col_hex)

    # Larghezze colonne legenda (sovrascrive solo colonne 1-8)
    largh_leg = [20, 16, 16, 20, 20, 16, 16, 20]
    for col_i, w in enumerate(largh_leg, start=1):
        ws.column_dimensions[get_column_letter(col_i)].width = w


# ── SALVATAGGIO PRINCIPALE ────────────────────────────────────────

def salva_excel_modalita_c(runs, metriche, nome_audio, trascrizioni_runs=None):
    """Salva il file Excel completo della Modalita C per un singolo audio."""
    nome_base = os.path.splitext(nome_audio)[0]
    nome_safe = re.sub(r'[<>:"/\\|?*]', '_', nome_base)
    file_output = os.path.join(OUTPUT_DIR, f"report_qualita_{nome_safe}.xlsx")

    frames = _runs_a_dataframes(runs)
    df_metriche = pd.DataFrame(metriche)

    with pd.ExcelWriter(file_output, engine="openpyxl") as writer:
        for sheet_name, df in frames.items():
            df.to_excel(writer, sheet_name=sheet_name, index=False)
        df_metriche.to_excel(writer, sheet_name="ANALISI QUALITA", index=False)

        wb = writer.book
        for name in wb.sheetnames:
            largh = _LARGHEZZE_C.get(name)
            if name == "ANALISI QUALITA":
                _applica_stile_qualita(wb[name], largh)
            else:
                _applica_stile(wb[name], largh)

        _aggiungi_foglio_trascrizioni(wb, runs, trascrizioni_runs)

    print(f"   Salvato: '{file_output}'")
    return file_output


print("Funzioni salvataggio Modalita C pronte.")
for nome, (s1, s2, s3, verso, gruppo, desc) in _SOGLIE_METRICHE.items():
    tag = "[REPORT]" if gruppo == "report" else "[TRASC] "
    if verso == "alto":
        print(f"   {tag} {nome:30s}  Verde>={s1}  Giallo>={s2}  Arancio>={s3}  Rosso<{s3}")
    else:
        print(f"   {tag} {nome:30s}  Verde<={s1}  Giallo<={s2}  Arancio<={s3}  Rosso>{s3}")


Funzioni salvataggio Modalita C pronte.
Metriche con colorazione semaforo:
   Cohen's Kappa (globale)         Verde>=0.81  Giallo>=0.61  Arancio>=0.4  Rosso<0.4
   Precision                       Verde>=0.85  Giallo>=0.7  Arancio>=0.5  Rosso<0.5
   Recall                          Verde>=0.85  Giallo>=0.7  Arancio>=0.5  Rosso<0.5
   F1 Score                        Verde>=0.85  Giallo>=0.7  Arancio>=0.5  Rosso<0.5
   Consistenza Lessicale           Verde>=0.75  Giallo>=0.55  Arancio>=0.35  Rosso<0.35
   CER                             Verde<=0.05  Giallo<=0.15  Arancio<=0.3  Rosso>0.3
   WER                             Verde<=0.1  Giallo<=0.25  Arancio<=0.4  Rosso>0.4
   DTA                             Verde>=0.9  Giallo>=0.75  Arancio>=0.5  Rosso<0.5
   EPR                             Verde<=0.1  Giallo<=0.25  Arancio<=0.4  Rosso>0.4


In [20]:
def _runs_a_dataframes(runs):
    """
    Converte la lista di run in DataFrame per ogni sezione,
    con 'Estrazione #' come prima colonna.
    """
    frames = {}

    # ── EVENTO ────────────────────────────────────────────────────
    righe_ev = []
    for r in runs:
        # Protezione: r["dati"] potrebbe essere una lista anziché un dict
        dati = r["dati"] if isinstance(r["dati"], dict) else {}
        ev = dati.get("Evento", {})
        righe_ev.append({
            "Estrazione #":          r["estrazione"],
            "ID Evento":             ev.get("id_evento", "N/D"),
            "Data / Ora":            ev.get("Data_Ora", "N/D"),
            "Tipo Evento":           ev.get("Tipo_Evento", "N/D"),
            "Gravità":               ev.get("Gravita_Stimata", "N/D"),
            "Situazione Pre-Evento": ev.get("Prima_dell_evento", "N/D"),
            "Trigger Principale":    ev.get("Trigger_Evento_Principale", "N/D"),
            "Dinamica (Sintesi)":    ev.get("Dinamica_Sintesi", "N/D"),
        })
    frames["EVENTO"] = pd.DataFrame(righe_ev)

    # ── LUOGHI & MEZZI ────────────────────────────────────────────
    righe_lm = []
    for r in runs:
        dati = r["dati"] if isinstance(r["dati"], dict) else {}
        lm = dati.get("Luoghi_e_Mezzi", {})
        righe_lm.append({
            "Estrazione #":          r["estrazione"],
            "Reparto":               lm.get("Reparto", "N/D"),
            "Area Coinvolta":        lm.get("Area_Coinvolta", "N/D"),
            "Macchinario":           lm.get("Macchinario_Coinvolto", "N/D"),
            "Codice Macchinario":    lm.get("Codice_Macchinario", "N/D"),
            "Condizioni Ambientali": lm.get("Condizioni_Ambientali", "N/D"),
        })
    frames["LUOGHI & MEZZI"] = pd.DataFrame(righe_lm)

    # ── ANALISI CAUSE ─────────────────────────────────────────────
    righe_ac = []
    for r in runs:
        dati = r["dati"] if isinstance(r["dati"], dict) else {}
        ac = dati.get("Analisi_Cause", {})
        righe_ac.append({
            "Estrazione #":              r["estrazione"],
            "Causa Tecnica":             ac.get("Causa_Tecnica", "N/D"),
            "Fattore Umano":             ac.get("Fattore_Umano", "N/D"),
            "Fattore Ambientale":        ac.get("Fattore_Ambientale", "N/D"),
            "Causa Radice (Root Cause)": ac.get("Causa_Radice", "N/D"),
        })
    frames["ANALISI CAUSE"] = pd.DataFrame(righe_ac)

    # ── CONSEGUENZE ───────────────────────────────────────────────
    righe_co = []
    for r in runs:
        dati = r["dati"] if isinstance(r["dati"], dict) else {}
        co = dati.get("Conseguenze", {})
        righe_co.append({
            "Estrazione #":               r["estrazione"],
            "Conseguenza Fisica":         co.get("Conseguenza_Fisica_Immediata", "N/D"),
            "Parte del Corpo Lesa":       co.get("Parte_Corpo_Lesa", "N/D"),
            "Giorni Inabilità (Stimati)": co.get("Giorni_Inabilita_Stimati", "N/D"),
            "Conseguenza Materiale":      co.get("Conseguenza_Materiale_Aziendale", "N/D"),
            "Fermo Produzione (ore)":     co.get("Fermo_Produzione_Ore", "N/D"),
        })
    frames["CONSEGUENZE"] = pd.DataFrame(righe_co)

    # ── AZIONI CORRETTIVE ─────────────────────────────────────────
    righe_az = []
    for r in runs:
        dati = r["dati"] if isinstance(r["dati"], dict) else {}
        az = dati.get("Azioni_Correttive", {})
        righe_az.append({
            "Estrazione #":                       r["estrazione"],
            "Procedura Violata":                  az.get("Procedura_Corretta_Violata", "N/D"),
            "DPI Mancanti/Inadeguati":            az.get("DPI_Mancanti_o_Inadeguati", "N/D"),
            "Azione Correttiva Immediata":        az.get("Azione_Correttiva_Immediata", "N/D"),
            "Azione Correttiva da Implementare":  az.get("Azione_Correttiva_da_Implementare", "N/D"),
            "Responsabile":                       az.get("Responsabile_Azione", "N/D"),
            "Scadenza":                           az.get("Scadenza_Azione", "N/D"),
        })
    frames["AZIONI CORRETTIVE"] = pd.DataFrame(righe_az)

    # ── PERSONE COINVOLTE ─────────────────────────────────────────
    righe_pc = []
    for r in runs:
        dati = r["dati"] if isinstance(r["dati"], dict) else {}
        pc = dati.get("Persone_Coinvolte", [])
        if isinstance(pc, list) and pc:
            for persona in pc:
                # Protezione: verificare che persona sia un dict, non una lista
                if isinstance(persona, dict):
                    row = {"Estrazione #": r["estrazione"]}
                    row.update({k.replace("_", " "): v for k, v in persona.items()})
                    righe_pc.append(row)
        else:
            righe_pc.append({"Estrazione #": r["estrazione"]})
    frames["PERSONE COINVOLTE"] = pd.DataFrame(righe_pc).fillna("N/D")

    return frames

In [22]:
# =====================================================================
# CELLA 19: ESEGUI — MODALITA C (analisi multi-run per ogni audio)
# Assicurati di aver eseguito prima:
#   - Cella 15 (configurazione N_ESTRAZIONI)
#   - Cella 16 (trascrizione multi-run, opzionale)
#   - Cella 17 (funzioni metriche)
#   - Cella 18 (funzioni salvataggio)
#   - Cella 19 (_runs_a_dataframes)
# =====================================================================

mancanti_c = [f for f in FILE_AUDIO if not os.path.exists(f)]
if mancanti_c:
    print(f"File mancanti: {mancanti_c}")
else:
    risultati_c  = {}
    file_salvati_c = []

    for percorso in FILE_AUDIO:
        nome = os.path.basename(percorso)
        print(f"\n{'='*60}")
        print(f"Elaborazione Modalita C: {nome}")
        print(f"{'='*60}")

        # 1. N estrazioni JSON
        runs = analizza_modalita_c(percorso, N_ESTRAZIONI)

        # 2. Trascrizioni multi-run (lista di N stringhe, una per run)
        trasc_runs = None
        try:
            trasc_runs = trascrizioni_c.get(nome, None)
        except NameError:
            pass   # Cella 16 non eseguita

        # 3. Ground truth testuale opzionale
        gt_testo = GROUND_TRUTH_TRASCRIZIONI.get(nome, "")

        # 4. Calcolo metriche (passa la lista di trascrizioni)
        print(f"\nCalcolo metriche qualita...")
        metriche = calcola_metriche_c(
            runs,
            trascrizioni_runs=trasc_runs,
            ground_truth_testo=gt_testo,
            nome_audio=nome
        )

        risultati_c[nome] = {"runs": runs, "metriche": metriche}

        # 5. Salvataggio Excel (passa la lista di trascrizioni per il foglio richtext)
        file_out = salva_excel_modalita_c(
            runs, metriche, nome,
            trascrizioni_runs=trasc_runs
        )
        file_salvati_c.append(file_out)

        # 6. Anteprima metriche
        display(Markdown(f"### Metriche qualita — {nome}"))
        df_preview = pd.DataFrame(metriche)
        display(df_preview.drop(columns=["Audio"], errors="ignore"))

    print(f"\n{'='*60}")
    print(f"Modalita C completata!")
    print(f"   {len(file_salvati_c)} file Excel in '{OUTPUT_DIR}/':")
    for f in file_salvati_c:
        print(f"   {f}")
    print(f"{'='*60}")



Elaborazione Modalita C: Elena RSPP.mp3

Modalita C — 'Elena RSPP.mp3' (3 estrazioni)
   Estrazione 1/3...
   ⬆️  Upload: Elena RSPP.mp3
   ⏳ Attendo elaborazione ✅
   Estrazione 1 completata.
   Estrazione 2/3...
   ⬆️  Upload: Elena RSPP.mp3
   ⏳ Attendo elaborazione ✅
   Estrazione 2 completata.
   Estrazione 3/3...
   ⬆️  Upload: Elena RSPP.mp3
   ⏳ Attendo elaborazione ✅
   Estrazione 3 completata.

Calcolo metriche qualita...
   Salvato: 'output_report\report_qualita_Elena RSPP.xlsx'


### Metriche qualita — Elena RSPP.mp3

,Estrazione #,Cohen's Kappa (globale),Precision,Recall,F1 Score,Consistenza Lessicale,CER,WER,DTA,EPR
0,1,1.0,0.667,0.857,0.750,0.252,N/D (no ground truth),N/D (no ground truth),0.1111,0.0
1,2,1.0,0.889,0.889,0.889,0.252,N/D (no ground truth),N/D (no ground truth),0.1111,0.0
2,3,1.0,0.833,0.556,0.667,0.252,N/D (no ground truth),N/D (no ground truth),0.1111,0.0
3,MEDIA,1.0,0.796,0.767,0.769,0.252,N/D,N/D,0.1110,0.0



Elaborazione Modalita C: Marco Capolinea.mp3

Modalita C — 'Marco Capolinea.mp3' (3 estrazioni)
   Estrazione 1/3...
   ⬆️  Upload: Marco Capolinea.mp3
   ⏳ Attendo elaborazione ✅
   Estrazione 1 completata.
   Estrazione 2/3...
   ⬆️  Upload: Marco Capolinea.mp3
   ⏳ Attendo elaborazione ✅
   Estrazione 2 completata.
   Estrazione 3/3...
   ⬆️  Upload: Marco Capolinea.mp3
   ⏳ Attendo elaborazione ✅
   Estrazione 3 completata.

Calcolo metriche qualita...
   Salvato: 'output_report\report_qualita_Marco Capolinea.xlsx'


### Metriche qualita — Marco Capolinea.mp3

,Estrazione #,Cohen's Kappa (globale),Precision,Recall,F1 Score,Consistenza Lessicale,CER,WER,DTA,EPR
0,1,1.0,0.750,1.0,0.857,0.659,N/D (no ground truth),N/D (no ground truth),0.0556,0.125
1,2,1.0,0.375,1.0,0.545,0.659,N/D (no ground truth),N/D (no ground truth),0.0556,0.000
2,3,1.0,1.000,1.0,1.000,0.659,N/D (no ground truth),N/D (no ground truth),0.0556,0.125
3,MEDIA,1.0,0.708,1.0,0.801,0.659,N/D,N/D,0.0560,0.083



Elaborazione Modalita C: Martina Operatrice.mp3

Modalita C — 'Martina Operatrice.mp3' (3 estrazioni)
   Estrazione 1/3...
   ⬆️  Upload: Martina Operatrice.mp3
   ⏳ Attendo elaborazione ✅
   Estrazione 1 completata.
   Estrazione 2/3...
   ⬆️  Upload: Martina Operatrice.mp3
   ⏳ Attendo elaborazione ✅
   Estrazione 2 completata.
   Estrazione 3/3...
   ⬆️  Upload: Martina Operatrice.mp3
   ⏳ Attendo elaborazione ✅
   Estrazione 3 completata.

Calcolo metriche qualita...
   Salvato: 'output_report\report_qualita_Martina Operatrice.xlsx'


### Metriche qualita — Martina Operatrice.mp3

,Estrazione #,Cohen's Kappa (globale),Precision,Recall,F1 Score,Consistenza Lessicale,CER,WER,DTA,EPR
0,1,1.0,0.750,0.667,0.706,0.511,N/D (no ground truth),N/D (no ground truth),0.1111,0.0000
1,2,1.0,0.429,0.429,0.429,0.511,N/D (no ground truth),N/D (no ground truth),0.1111,0.0000
2,3,1.0,0.727,1.000,0.842,0.511,N/D (no ground truth),N/D (no ground truth),0.1111,0.1818
3,MEDIA,1.0,0.635,0.699,0.659,0.511,N/D,N/D,0.1110,0.0610



Modalita C completata!
   3 file Excel in 'output_report/':
   output_report\report_qualita_Elena RSPP.xlsx
   output_report\report_qualita_Marco Capolinea.xlsx
   output_report\report_qualita_Martina Operatrice.xlsx
